Import Lib

In [119]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
import requests

##### Tool 1

In [120]:
@tool
def get_conversion_factor(base_currency:str, target_currency:str)->float:
  """
  This function fetched the currency conversion factor between a given base currency and a target currency
  """
  
  url = f'https://v6.exchangerate-api.com/v6/1db4974fb42b289d9d2f9dea/pair/{base_currency}/{target_currency}'
  response = requests.get(url)

  return response.json()

In [121]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1756339201,
 'time_last_update_utc': 'Thu, 28 Aug 2025 00:00:01 +0000',
 'time_next_update_unix': 1756425601,
 'time_next_update_utc': 'Fri, 29 Aug 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 87.7256}

##### Tool 2

Bescause the LLM have already trained on the past data , so the prompt is to 
convert the cureency in to another so , they will firstly segment the "base_currency" & "target_currency". Now the LLM have data when it's trained so the tool got call and get the previous value.

To takle this Langchain have the method -> Annotated

In [122]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

when the LLM call the convert , they don't set the conversion rate. "I (developer) will inject this value after running eariler tools"

In [123]:
@tool
def convert(base_currency_value:int, conversion_rate: Annotated[float, InjectedToolArg])->float:
  """
  We have the currency conversion rate this function calculates the target currency value from a given base currency value
  """
  return base_currency_value * conversion_rate

In [124]:
convert.invoke({'base_currency_value':10, 'conversion_rate':87.7256})

877.256

### Tool Binding

In [125]:
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.7
)

In [126]:
llm_with_tool = llm.bind_tools([get_conversion_factor, convert])

In [127]:
messages = [
  HumanMessage(content="what is the conversion factor betweeen USD and INR , and based on that you can convert 10 USD in to INR")
]
messages

[HumanMessage(content='what is the conversion factor betweeen USD and INR , and based on that you can convert 10 USD in to INR', additional_kwargs={}, response_metadata={})]

### Tool Calling

In [128]:
llm_message = llm_with_tool.invoke(messages)
messages.append(llm_message)
llm_message

AIMessage(content='', additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"base_currency_value": 10.0}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--ab84fb0b-60d9-42cc-a1bc-06e7f7d4cf93-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': '0389d8cd-c4f0-4052-a075-9770fbdc91a7', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10.0}, 'id': 'a7ba8e0d-1c67-455c-8fe7-f3b093315897', 'type': 'tool_call'}], usage_metadata={'input_tokens': 94, 'output_tokens': 20, 'total_tokens': 114, 'input_token_details': {'cache_read': 0}})

only set the "base_currency", the "conversion_rate" not set because of annotated

In [129]:
llm_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'INR', 'base_currency': 'USD'},
  'id': '0389d8cd-c4f0-4052-a075-9770fbdc91a7',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10.0},
  'id': 'a7ba8e0d-1c67-455c-8fe7-f3b093315897',
  'type': 'tool_call'}]

### Tool Execute

In [130]:
import json

for tool_call in llm_message.tool_calls:

  # execute the 1st tool and get the value of the conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    
    # fetch the conversion_rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message in the message list (conerted json in to dict)
    messages.append(tool_message1)
  
  
  # execute the 2nd tool and get the conversion rate from tool 1
  if tool_call['name'] == 'convert':

    # fetch the current args
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2  = convert.invoke(tool_call)
    # append this tool message in the message list
    messages.append(tool_message2)



In [131]:
messages

[HumanMessage(content='what is the conversion factor betweeen USD and INR , and based on that you can convert 10 USD in to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"base_currency_value": 10.0}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--ab84fb0b-60d9-42cc-a1bc-06e7f7d4cf93-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': '0389d8cd-c4f0-4052-a075-9770fbdc91a7', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10.0, 'conversion_rate': 87.7256}, 'id': 'a7ba8e0d-1c67-455c-8fe7-f3b093315897', 'type': 'tool_call'}], usage_metadata={'input_tokens': 94, 'output_tokens': 20, 'total_tokens': 114, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='{"result": "success", "documentation": "https:

In [132]:
llm_with_tool.invoke(messages).content

'The conversion factor between USD and INR is 87.7256.  10 USD is equal to 877.26 INR.'